In [ ]:
# ============================================================================
#  Construction du tableau QSAR : SMILES + matrice de COMPTAGE des TOP_K ECFP
#  les plus fréquents (rayon >= MIN_RADIUS uniquement) + HOMO / LUMO / GAP
# ============================================================================

import csv
import numpy as np
import pandas as pd
from collections import Counter
from tqdm import tqdm
from rdkit import Chem, RDLogger
from rdkit.Chem import rdMolDescriptors

RDLogger.DisableLog("rdApp.*")

# ---- Configuration ---------------------------------------------------------
CSV_PATH    = "../01_datasets/molecules_PubChemQC.csv"
RADIUS      = 2          # rayon Morgan : 2 = ECFP4, 3 = ECFP6, 4 = ECFP8
MIN_RADIUS  = 0          # on exclut les bits de rayon < 0 (atomes isolés, paires)
TOP_K       = 200
LIMIT       = 1000000
OUT_PARQUET = "ecfp4_top1000_dataset.parquet"
OUT_BITS    = "ecfp4_top1000_bits.csv"


def iter_rows(path, limit=None):
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            if limit is not None and i >= limit:
                break
            def _f(x):
                try: return float(x)
                except (TypeError, ValueError): return np.nan
            yield (row["SMILES"],
                   _f(row["DFT:HOMO_ENERGY"]),
                   _f(row["DFT:LUMO_ENERGY"]),
                   _f(row["DFT:HOMO_LUMO_GAP"]))


def ecfp4_counts(smiles):
    """Dict {bit_id: count} pour les bits de rayon >= MIN_RADIUS seulement."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    bi = {}
    fp = rdMolDescriptors.GetMorganFingerprint(mol, RADIUS, bitInfo=bi)
    counts = fp.GetNonzeroElements()
    return {bid: cnt for bid, cnt in counts.items()
            if bi[bid][0][1] >= MIN_RADIUS}


In [ ]:
# ============================================================================
#  CHARGEMENT RAPIDE — saute les deux passes de calcul ECFP
#  Lancer UNIQUEMENT cette cellule suffit pour reprendre à partir des analyses
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from rdkit import Chem, RDLogger
from rdkit.Chem import rdMolDescriptors

RDLogger.DisableLog("rdApp.*")

OUT_PARQUET = "ecfp4_top1000_dataset.parquet"
OUT_BITS    = "ecfp4_top1000_bits.csv"

RADIUS = 2

df      = pd.read_parquet(OUT_PARQUET)
bits_df = pd.read_csv(OUT_BITS)

# Variables attendues par les cellules suivantes
bit_cols   = [c for c in df.columns if c.startswith("ECFP4_")]
top_ids    = [int(c.replace("ECFP4_", "")) for c in bit_cols]
col_index  = {bid: j for j, bid in enumerate(top_ids)}
ecfp_cols  = bit_cols
smiles_col = df["SMILES"].values

print(f"df chargé          : {df.shape[0]:,} molécules × {df.shape[1]} colonnes")
print(f"bits_df chargé     : {len(bits_df)} sous-structures")
print(f"Colonnes ECFP4     : {len(ecfp_cols)}")
print(f"Cibles disponibles : HOMO, LUMO, GAP")


In [ ]:
# ---- PASSE 1 : fréquence documentaire de chaque sous-structure -------------
# (dans combien de molécules chaque sous-structure ECFP4 apparaît-elle)
print("Passe 1/2 — comptage des sous-structures ECFP4 ...")
doc_freq = Counter()
n_rows = 0
n_bad  = 0
for smiles, *_ in tqdm(iter_rows(CSV_PATH, LIMIT), unit=" mol"):
    n_rows += 1
    counts = ecfp4_counts(smiles)
    if counts is None:
        n_bad += 1
        continue
    # fréquence documentaire : on compte une fois par molécule, peu importe
    # le nombre d'occurrences dans la molécule
    doc_freq.update(counts.keys())

print(f"  molécules lues        : {n_rows:,}")
print(f"  SMILES non lisibles   : {n_bad:,}")
print(f"  sous-structures unique: {len(doc_freq):,}")

# ---- Sélection des TOP_K sous-structures les plus présentes ----------------
# tri : fréquence décroissante, puis id croissant (résultat déterministe)
top = sorted(doc_freq.items(), key=lambda kv: (-kv[1], kv[0]))[:TOP_K]
top_ids   = [bit_id for bit_id, _ in top]
col_index = {bit_id: j for j, bit_id in enumerate(top_ids)}
bit_cols  = [f"ECFP4_{bit_id}" for bit_id in top_ids]
print(f"  -> {len(top_ids)} sous-structures retenues "
      f"(de {top[0][1]:,} à {top[-1][1]:,} molécules)")

# table de référence des bits retenus
bits_df = pd.DataFrame({
    "rank":      np.arange(1, len(top_ids) + 1),
    "ecfp4_id":  top_ids,
    "doc_count": [c for _, c in top],
    "fraction":  [c / n_rows for _, c in top],
})

# ---- PASSE 2 : matrice de COMPTAGE + métadonnées --------------------------
# uint16 : autorise jusqu'à 65 535 occurrences d'une même sous-structure
# dans une seule molécule (largement suffisant)
print("Passe 2/2 — construction de la matrice de comptage ...")
counts_matrix = np.zeros((n_rows, len(top_ids)), dtype=np.uint16)
smiles_col = np.empty(n_rows, dtype=object)
homo = np.empty(n_rows, dtype=np.float64)
lumo = np.empty(n_rows, dtype=np.float64)
gap  = np.empty(n_rows, dtype=np.float64)

for i, (smiles, h, l, g) in enumerate(
        tqdm(iter_rows(CSV_PATH, LIMIT), total=n_rows, unit=" mol")):
    smiles_col[i] = smiles
    homo[i], lumo[i], gap[i] = h, l, g
    counts = ecfp4_counts(smiles)
    if counts is None:
        continue
    for bit_id, n in counts.items():
        j = col_index.get(bit_id)
        if j is not None:
            counts_matrix[i, j] = n         # ← VRAI COMPTE, pas 0/1

# ---- Assemblage du tableau final ------------------------------------------
df = pd.DataFrame(counts_matrix, columns=bit_cols, copy=False)
df.insert(0, "SMILES", smiles_col)
df["HOMO"] = homo
df["LUMO"] = lumo
df["GAP"]  = gap

print(f"\nTableau final : {df.shape[0]:,} lignes x {df.shape[1]:,} colonnes")
print(f"Valeur max dans la matrice de comptage : {counts_matrix.max()}")
print(f"Nb de cellules > 1 : {(counts_matrix > 1).sum():,} "
      f"({(counts_matrix > 1).sum() / counts_matrix.size * 100:.2f} %)")
df.head()


In [ ]:
# ---- Sauvegarde ------------------------------------------------------------
# Parquet : compact et rapide à relire (df.read_parquet). Le CSV serait
# énorme (~1000 colonnes x ~880k lignes) et lent ; à éviter ici.
df.to_parquet(OUT_PARQUET, index=False)
bits_df.to_csv(OUT_BITS, index=False)

print(f"Tableau    -> {OUT_PARQUET}")
print(f"Bits ECFP4 -> {OUT_BITS}")
print("\nAperçu des sous-structures les plus présentes :")
bits_df.head(10)


In [ ]:
# ============================================================================
#  Influence of ECFP4 bits on HOMO, LUMO and GAP
#  Absolute Pearson correlation |r| (point-biserial for binary variable)
# ============================================================================
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap

TARGET_COLORS = {"HOMO": "#F472B6", "LUMO": "#7B52AB", "GAP": "#4472C4"}
TARGET_CMAPS  = {
    t: LinearSegmentedColormap.from_list(f"cm_{t}", ["#ffffff", TARGET_COLORS[t]])
    for t in TARGET_COLORS
}

ecfp_cols = [c for c in df.columns if c.startswith("ECFP4_")]

mask = df[["HOMO", "LUMO", "GAP"]].notna().all(axis=1)
df_c = df[mask].reset_index(drop=True)
print(f"Molecules used: {len(df_c):,}  ({mask.mean()*100:.1f} %)\n")

TOP = 20

corr        = {}   # |r| trié desc  — pour les graphiques / sélection top bits
corr_signed = {}   # r signé        — pour connaître le sens de l'effet
for target in ("HOMO", "LUMO", "GAP"):
    raw = df_c[ecfp_cols].corrwith(df_c[target])
    corr_signed[target] = raw
    corr[target]        = raw.abs().sort_values(ascending=False)

for target, series in corr.items():
    top = series.head(TOP).reset_index()
    top.columns = ["ECFP4", f"|r| with {target}"]
    top.index = range(1, TOP + 1)
    print(f"{'━'*54}")
    print(f"  Top {TOP} ECFP4  —  influence on {target}")
    print(f"{'━'*54}")
    try:
        display(top)
    except NameError:
        print(top.to_string())
    print()

fig, axes = plt.subplots(1, 3, figsize=(20, 9), facecolor="white")
fig.suptitle("Influence of ECFP4 substructures on HOMO, LUMO and GAP",
             fontsize=14, fontweight="bold", y=1.01)

for ax, (target, series) in zip(axes, corr.items()):
    top    = series.head(TOP)
    labels = [c.replace("ECFP4_", "") for c in top.index]
    ypos   = range(TOP, 0, -1)
    colors = TARGET_CMAPS[target](np.linspace(0.35, 1.0, TOP))

    bars = ax.barh(list(ypos), top.values, color=colors, alpha=0.90)

    ax.set_yticks(list(ypos))
    ax.set_yticklabels(labels, fontsize=7.5)
    ax.set_xlabel("|Pearson correlation|", fontsize=10)
    ax.set_title(f"Top {TOP} ECFP4  →  {target}",
                 fontsize=13, fontweight="bold", color=TARGET_COLORS[target])
    ax.set_xlim(0, top.values[0] * 1.15)
    ax.set_facecolor("white")

    for bar, val in zip(bars, top.values):
        ax.text(val + top.values[0] * 0.015,
                bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=6.5)

plt.tight_layout()
plt.savefig("ecfp4_influence_top30.png", dpi=300, bbox_inches="tight",
            facecolor="white")
plt.show()
print("Figure -> ecfp4_influence_top30.png")


In [ ]:
# ============================================================================
#  Concrete effect of ECFP4 bits on HOMO / LUMO / GAP
#  Top 6 bits per target: boxplot des molécules où le bit est PRÉSENT
#  + ligne pointillée = moyenne globale de la cible
# ============================================================================
import matplotlib.colors as mcolors

TARGET_COLORS = {"HOMO": "#F472B6", "LUMO": "#7B52AB", "GAP": "#4472C4"}

N_BOX = 6

df_box = df_c[(df_c["HOMO"] >= -15) & (df_c["LUMO"] >= -15)].reset_index(drop=True)
print(f"Molecules used for box plots: {len(df_box):,}  "
      f"({len(df_c)-len(df_box)} DFT outliers removed)")

fig, axes = plt.subplots(3, N_BOX, figsize=(18, 10), facecolor="white")
fig.suptitle("Distribution of HOMO / LUMO / GAP when ECFP4 bit is present\n"
             "(dashed line = global mean)",
             fontsize=13, fontweight="bold", y=1.02)

for row_i, (target, ser) in enumerate(corr.items()):
    top_bits   = ser.head(N_BOX).index.tolist()
    base_col   = TARGET_COLORS[target]
    global_mean = df_box[target].mean()

    for col_i, feat in enumerate(top_bits):
        ax = axes[row_i, col_i]

        present = df_box.loc[df_box[feat] > 0, target].dropna()

        bp = ax.boxplot(
            [present],
            tick_labels=["present"],
            patch_artist=True,
            widths=0.5,
            medianprops=dict(color="white", linewidth=2.0),
            whiskerprops=dict(color="#4a4a4a", lw=1.2),
            capprops=dict(color="#4a4a4a", lw=1.2),
            flierprops=dict(marker="o", color=base_col, markersize=2.5,
                            alpha=0.4, linestyle="none"),
        )
        bp["boxes"][0].set_facecolor(mcolors.to_rgba(base_col, alpha=0.75))
        bp["boxes"][0].set_edgecolor("#4a4a4a")
        bp["boxes"][0].set_linewidth(1.0)

        ax.axhline(global_mean, linestyle="--", color="#555", linewidth=1.2,
                   zorder=5)

        ax.set_facecolor("white")
        delta = present.median() - global_mean
        ax.set_title(f"{feat.replace('ECFP4_', '')}\nΔmed={delta:+.2f} eV",
                     fontsize=7.5)
        ax.tick_params(axis="x", labelsize=7)
        if col_i == 0:
            ax.set_ylabel(target, fontsize=10, fontweight="bold",
                          color=TARGET_COLORS[target])
        ax.yaxis.set_tick_params(labelsize=7)

plt.tight_layout()
plt.savefig("ecfp4_effect_boxplots.png", dpi=300, bbox_inches="tight",
            facecolor="white")
plt.show()
print("Figure -> ecfp4_effect_boxplots.png")


In [ ]:
# ============================================================================
#  Ridge regression for HOMO, LUMO and GAP — equations + predicted vs actual
# ============================================================================
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.axes_grid1.inset_locator import inset_axes as _inset_axes

N_FORMULA    = 15
MIN_ECFP_BITS = 3   # molécules avec moins de MIN_ECFP_BITS bits présents exclues

TARGET_COLORS = {"HOMO": "#F472B6", "LUMO": "#7B52AB", "GAP": "#4472C4"}

cmap_density = LinearSegmentedColormap.from_list(
    "candy_pop", ["#e9f6ff", "#f0d7ff", "#cd9cf7b3", "#f465bf", "#c71585"])

def local_density(x, y, bins=60):
    H, x_e, y_e = np.histogram2d(x, y, bins=bins)
    xi = np.clip(np.digitize(x, x_e) - 1, 0, bins - 1)
    yi = np.clip(np.digitize(y, y_e) - 1, 0, bins - 1)
    return H[xi, yi]

def _marginal_hist(ax_ref, ax_inset, values, orientation, n_bins=30):
    counts, edges = np.histogram(values, bins=n_bins)
    centers = 0.5 * (edges[:-1] + edges[1:])
    widths  = edges[1:] - edges[:-1]
    if orientation == "y":
        ax_inset.barh(centers, counts, height=widths * 0.9,
                      color="#e8cdffb3", linewidth=0, alpha=0.9)
        ax_inset.invert_xaxis()
        ax_inset.set_ylim(ax_ref.get_ylim())
    else:
        ax_inset.bar(centers, counts, width=widths * 0.9,
                     color="#e8cdffb3", linewidth=0, alpha=0.9)
        ax_inset.set_xlim(ax_ref.get_xlim())
    ax_inset.set_xticks([]); ax_inset.set_yticks([])
    for sp in ax_inset.spines.values():
        sp.set_visible(False)
    ax_inset.patch.set_alpha(0.0)

# ---- Filtres : DFT outliers + au moins MIN_ECFP_BITS bits présents ----------
before = len(df_c)
df_reg = df_c[(df_c["HOMO"] >= -15) & (df_c["LUMO"] >= -15)].reset_index(drop=True)
n_bits_present = (df_reg[ecfp_cols] > 0).sum(axis=1)
df_reg = df_reg[n_bits_present >= MIN_ECFP_BITS].reset_index(drop=True)

print(f"Molecules avant filtre       : {before:,}")
print(f"Après filtre DFT outliers    : {(df_c['HOMO'] >= -15).sum():,}")
print(f"Après filtre ECFP ≥ {MIN_ECFP_BITS} bits  : {len(df_reg):,}  "
      f"({before - len(df_reg)} retirées au total)\n")

X = df_reg[ecfp_cols].values.astype(np.float32)

TARGETS = ["HOMO", "LUMO", "GAP"]

# ---- Training --------------------------------------------------------------
results = {}
for target in TARGETS:
    y = df_reg[target].values
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    model = Ridge(alpha=1.0)
    model.fit(X_tr, y_tr)
    y_pred_tr = model.predict(X_tr)
    y_pred_te = model.predict(X_te)
    results[target] = dict(
        model=model,
        X_tr=X_tr, X_te=X_te,
        y_tr=y_tr, y_te=y_te,
        y_pred_tr=y_pred_tr, y_pred_te=y_pred_te,
        r2_tr=r2_score(y_tr, y_pred_tr),
        r2_te=r2_score(y_te, y_pred_te),
        rmse_tr=np.sqrt(mean_squared_error(y_tr, y_pred_tr)),
        rmse_te=np.sqrt(mean_squared_error(y_te, y_pred_te)),
        mae_tr=mean_absolute_error(y_tr, y_pred_tr),
        mae_te=mean_absolute_error(y_te, y_pred_te),
    )

# ---- Equations -------------------------------------------------------------
for target in TARGETS:
    r     = results[target]
    m     = r["model"]
    order = np.argsort(np.abs(m.coef_))[::-1]
    print(f"{'='*60}")
    print(f"  Ridge equation — {target}  (alpha=1.0)")
    print(f"{'='*60}")
    print(f"  {target}_predicted = {m.intercept_:.4f}")
    for j in order[:N_FORMULA]:
        print(f"    {m.coef_[j]:+.4f}  x  {ecfp_cols[j]}")
    print(f"    ... + {len(ecfp_cols)-N_FORMULA} other terms")
    print(f"  R2   train={r['r2_tr']:.4f}  test={r['r2_te']:.4f}  |  "
          f"RMSE train={r['rmse_tr']:.4f}  test={r['rmse_te']:.4f} eV  |  "
          f"MAE  train={r['mae_tr']:.4f}  test={r['mae_te']:.4f} eV\n")

metrics_df = pd.DataFrame({
    "Target":     TARGETS,
    "R2 train":   [round(results[t]["r2_tr"],  4) for t in TARGETS],
    "R2 test":    [round(results[t]["r2_te"],  4) for t in TARGETS],
    "RMSE train": [round(results[t]["rmse_tr"],4) for t in TARGETS],
    "RMSE test":  [round(results[t]["rmse_te"],4) for t in TARGETS],
    "MAE train":  [round(results[t]["mae_tr"], 4) for t in TARGETS],
    "MAE test":   [round(results[t]["mae_te"], 4) for t in TARGETS],
}).set_index("Target")
try:
    display(metrics_df)
except NameError:
    print(metrics_df.to_string())

# ---- 3 x 2 scatter grid with density coloring ------------------------------
rng   = np.random.default_rng(0)
N_MAX = 10000000

fig, axes = plt.subplots(3, 2, figsize=(14, 18), facecolor="white")
fig.suptitle("Predicted vs Actual — Ridge ECFP4 (DFT outliers + ECFP < 3 bits removed)",
             fontsize=14, fontweight="bold", y=1.01)

col_titles = ["Train (80 %)", "Test (20 %)"]
for j, title in enumerate(col_titles):
    axes[0, j].set_title(title, fontsize=11, fontweight="bold", pad=12)

for row_i, target in enumerate(TARGETS):
    r = results[target]
    y_tr = r["y_tr"];  y_prd_tr = r["y_pred_tr"]
    y_te = r["y_te"];  y_prd_te = r["y_pred_te"]

    idx_tr = rng.choice(len(y_tr), size=min(N_MAX, len(y_tr)), replace=False)
    idx_te = rng.choice(len(y_te), size=min(N_MAX, len(y_te)), replace=False)

    all_v  = np.concatenate([y_tr, y_prd_tr, y_te, y_prd_te])
    lo, hi = all_v.min() - 0.3, all_v.max() + 0.3
    diag   = [lo, hi]

    for col_j, (y_real, y_pred, r2_val, rmse_val, mae_val) in enumerate([
        (y_tr[idx_tr], y_prd_tr[idx_tr], r["r2_tr"], r["rmse_tr"], r["mae_tr"]),
        (y_te[idx_te], y_prd_te[idx_te], r["r2_te"], r["rmse_te"], r["mae_te"]),
    ]):
        ax = axes[row_i, col_j]

        z     = local_density(y_real, y_pred, bins=40)
        order = np.argsort(z)
        sc    = ax.scatter(y_real[order], y_pred[order], c=z[order],
                           cmap=cmap_density, s=8, alpha=0.75, rasterized=True)
        plt.colorbar(sc, ax=ax, label="Density", pad=0.02, fraction=0.030)

        ax.plot(diag, diag, color=TARGET_COLORS[target], lw=1.8, zorder=5,
                label="predicted = actual")
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect("equal")
        ax.set_facecolor("white")

        ax.set_ylabel(f"{target} predicted (eV)", fontsize=10,
                      color=TARGET_COLORS[target], fontweight="bold")
        ax.set_xlabel(f"{target} actual (eV)", fontsize=9)

        note = (f"R2   = {r2_val:.4f}\n"
                f"RMSE = {rmse_val:.4f} eV\n"
                f"MAE  = {mae_val:.4f} eV\n"
                f"n    = {len(y_real):,}")
        ax.text(0.04, 0.97, note, transform=ax.transAxes,
                fontsize=8, va="top", family="monospace",
                bbox=dict(facecolor="white", alpha=0.85, edgecolor="none", pad=4))

        ax_y = _inset_axes(ax, width="18%", height="100%", loc="center right",
                           bbox_to_anchor=(0.0, 0, 1, 1),
                           bbox_transform=ax.transAxes, borderpad=0)
        _marginal_hist(ax, ax_y, y_pred, "y")

        ax_x = _inset_axes(ax, width="100%", height="18%", loc="lower center",
                           bbox_to_anchor=(0, 0, 1, 1),
                           bbox_transform=ax.transAxes, borderpad=0)
        _marginal_hist(ax, ax_x, y_real, "x")

plt.tight_layout()
plt.savefig("ecfp4_regression_HOMO_LUMO_GAP.png", dpi=300, bbox_inches="tight",
            facecolor="white")
plt.show()
print("Figure -> ecfp4_regression_HOMO_LUMO_GAP.png")

In [ ]:
import io
from tqdm.notebook import tqdm
from rdkit.Chem import Draw, AllChem
from IPython.display import display as ipy_display, Image as IPyImage
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image as PILImage

N_TOP        = 200
MOLS_PER_ROW = 5

# ---- Collecte des bits à illustrer -----------------------------------------
bits_needed = set()
for ser in corr.values():
    for bit_col in ser.head(N_TOP).index:
        bits_needed.add(int(bit_col.replace("ECFP4_", "")))
print(f"Bits à illustrer : {len(bits_needed)}")

# ---- Cache MINIMAL ---------------------------------------------------------
example_mol = {}
remaining   = set(bits_needed)

for smi in tqdm(smiles_col, desc="Recherche d'exemples", unit=" mol"):
    if not remaining:
        break
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    bi = {}
    rdMolDescriptors.GetMorganFingerprint(mol, RADIUS, bitInfo=bi)
    found = remaining & bi.keys()
    for bid in found:
        example_mol[bid] = (mol, bi)
    remaining -= found

print(f"✓ {len(example_mol)}/{len(bits_needed)} bits illustrés\n")

# ---- Figures par cible (DrawMorganEnvs — notation blog RDKit) ---------------
SUB_W, SUB_H = 360, 300

for target, ser in corr.items():
    print(f"\n{'━'*80}\n  Top {N_TOP} fragments  →  {target}\n{'━'*80}")
    rows, envs, legends = [], [], []

    for rank, (bit_col, abs_r) in enumerate(ser.head(N_TOP).items(), start=1):
        bid       = int(bit_col.replace("ECFP4_", ""))
        signed_r  = corr_signed[target].loc[bit_col]
        direction = "↑ +" if signed_r > 0 else "↓ −"
        n_occ     = int((df[bit_col] > 0).sum())

        rows.append({"Rang": rank, "r signé": round(signed_r, 4),
                     "effet": direction, "n mol": n_occ})

        if bid in example_mol:
            mol_ex, bi_ex = example_mol[bid]
            aix_ex, r_ex  = bi_ex[bid][0]
            sign_label    = f"+{signed_r:.3f}" if signed_r > 0 else f"{signed_r:.3f}"
            envs.append((mol_ex, aix_ex, r_ex))
            legends.append(f"#{rank}  r={sign_label}  n={n_occ:,}")

    sub = pd.DataFrame(rows); sub.index = range(1, len(sub)+1)
    try:    display(sub)
    except: print(sub.to_string())

    if envs:
        fname    = f"ecfp4_fragments_{target}.png"
        raw      = Draw.DrawMorganEnvs(
            envs, molsPerRow=MOLS_PER_ROW,
            subImgSize=(SUB_W, SUB_H), legends=legends,
            useSVG=False,
        )
        # DrawMorganEnvs(useSVG=False) renvoie des bytes PNG
        png_bytes = raw if isinstance(raw, bytes) else raw.data
        pil_img   = PILImage.open(io.BytesIO(png_bytes))

        # ---- Affichage matplotlib avec titre coloré -------------------------
        fig_w = pil_img.width  / 120
        fig_h = pil_img.height / 120 + 0.7
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), facecolor="white")
        ax.imshow(np.array(pil_img))
        ax.axis("off")
        ax.set_title(
            target, fontsize=18, fontweight="bold",
            color=TARGET_COLORS[target], fontfamily="Arial", pad=10
        )
        plt.tight_layout()
        plt.savefig(fname, dpi=300, bbox_inches="tight", facecolor="white")
        plt.show()
        print(f"Figure → {fname}")

**Figure — Top ECFP4 environments ranked by absolute Pearson correlation $|r|$ with HOMO, LUMO and GAP, computed on 882 924 molecules from PubChemQC.**  
Visualization inspired by the [RDKit blog post on chemical words](https://greglandrum.github.io/rdkit-blog/posts/2023-01-08-chemical-words.html) and produced with `Draw.DrawMorganEnvs`.

Each panel shows the **source molecule** in which the circular environment was first encountered, with the ECFP4 substructure highlighted directly inside that molecule context rather than as an isolated fragment.

**Color and line encoding (RDKit defaults):**

| Element | Color | Meaning |
|---|---|---|
| **Central atom** | Light blue (RGB 0.6, 0.8, 1.0) | Atom at the center of the Morgan fingerprint bit |
| **Aromatic bonds / atoms** | Pale yellow (RGB 0.9, 0.9, 0.2) | Aromatic system within the captured environment |
| **Ring atoms** | Light gray (RGB 0.8, 0.8, 0.8) | Non-aromatic ring membership inside the environment |
| **Outer environment atoms** | Very light gray (RGB 0.9, 0.9, 0.9) | Atoms at the periphery of the circular environment |
| **Non-highlighted atoms** | White / standard RDKit palette | Atoms outside the ECFP4 environment (context only) |
| **Bonds inside environment** | Thickened, colored stroke | Bonds within the highlighted substructure |
| **Bonds outside environment** | Thin gray stroke | Context bonds not part of the fingerprint bit |

The **legend** below each structure reports: rank `#`, signed Pearson correlation `r` (+ raises / − lowers the target energy), and `n` = number of molecules in the dataset containing that environment.

In [ ]:
# ============================================================================
#  Boxplots HOMO / LUMO / GAP + structure ECFP+contexte sous chaque boxplot
#  Deux séries de figures par cible :
#    - classées par |Δmédiane|  -> ecfp4_boxplots_delta_{target}.png
#    - classées par |r|         -> ecfp4_boxplots_r_{target}.png
# ============================================================================
import io
import math
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import numpy as np
from PIL import Image as PILImage
from rdkit.Chem import Draw

N_BOX        = 30
COLS_PER_ROW = 5
MIN_FREQ     = 100

N_ROWS_PER_TARGET = math.ceil(N_BOX / COLS_PER_ROW)  # 6 lignes de 5

df_box = df_c[(df_c["HOMO"] >= -15) & (df_c["LUMO"] >= -15)].reset_index(drop=True)
print(f"Molécules utilisées : {len(df_box):,}")

# --- Calcul du Δmédiane ----------------------------------------------------
print("Calcul des effets (Δmédiane) ...")
delta_median = {}
for target in ["HOMO", "LUMO", "GAP"]:
    global_med = df_box[target].median()
    deltas = {}
    for feat in ecfp_cols:
        mask = df_box[feat] > 0
        if mask.sum() < MIN_FREQ:
            continue
        deltas[feat] = df_box.loc[mask, target].median() - global_med
    delta_median[target] = pd.Series(deltas)

# --- Fonction de dessin -----------------------------------------------------
def draw_boxplot_figure(target, top_bits, fname, title):
    base_col   = TARGET_COLORS[target]
    global_med = df_box[target].median()
    height_ratios = [ratio for _ in range(N_ROWS_PER_TARGET) for ratio in [2.5, 1.6]]

    fig = plt.figure(
        figsize=(COLS_PER_ROW * 3.4, N_ROWS_PER_TARGET * 4.5),
        facecolor="white",
    )
    fig.suptitle(title, fontsize=13, fontweight="bold", y=1.01)

    inner = gridspec.GridSpec(
        2 * N_ROWS_PER_TARGET, COLS_PER_ROW,
        figure=fig,
        height_ratios=height_ratios,
        hspace=0.45, wspace=0.35,
    )

    for col_i, feat in enumerate(top_bits):
        grid_row = col_i // COLS_PER_ROW
        grid_col = col_i % COLS_PER_ROW

        bid       = int(feat.replace("ECFP4_", ""))
        present   = df_box.loc[df_box[feat] > 0, target].dropna()
        n_present = len(present)
        delta     = delta_median[target].get(feat, float("nan"))
        r_val     = corr_signed[target].get(feat, float("nan"))

        ax_box = fig.add_subplot(inner[grid_row * 2, grid_col])
        bp = ax_box.boxplot(
            [present], tick_labels=[""], patch_artist=True, widths=0.5,
            medianprops=dict(color="white", linewidth=2.0),
            whiskerprops=dict(color="#4a4a4a", lw=1.2),
            capprops=dict(color="#4a4a4a", lw=1.2),
            flierprops=dict(marker="o", color=base_col, markersize=2.0,
                            alpha=0.35, linestyle="none"),
        )
        bp["boxes"][0].set_facecolor(mcolors.to_rgba(base_col, alpha=0.75))
        bp["boxes"][0].set_edgecolor("#4a4a4a")
        ax_box.axhline(global_med, linestyle="--", color="#555", linewidth=1.2, zorder=5)
        ax_box.set_facecolor("white")
        ax_box.yaxis.set_tick_params(labelsize=7)
        if grid_col == 0:
            ax_box.set_ylabel("Energy (eV)", fontsize=9, color="#4a4a4a")

        ax_img = fig.add_subplot(inner[grid_row * 2 + 1, grid_col])
        if bid in example_mol:
            mol_ex, bi_ex = example_mol[bid]
            aix_ex, r_ex  = bi_ex[bid][0]
            try:
                raw = Draw.DrawMorganEnvs(
                    [(mol_ex, aix_ex, r_ex)],
                    molsPerRow=1, subImgSize=(210, 125),
                    useSVG=False,
                )
                png_bytes = raw if isinstance(raw, bytes) else raw.data
                img = PILImage.open(io.BytesIO(png_bytes))
                ax_img.imshow(np.array(img))
            except Exception:
                ax_img.text(0.5, 0.5, str(bid), ha="center", va="center",
                            fontsize=10, transform=ax_img.transAxes)
        else:
            ax_img.text(0.5, 0.5, "—", ha="center", va="center",
                        fontsize=10, transform=ax_img.transAxes)

        ax_img.set_xticks([])
        ax_img.set_yticks([])
        for sp in ax_img.spines.values():
            sp.set_visible(False)

        n_occ = int((df[feat] > 0).sum())
        ax_img.text(
            0.5, -0.05,
            f"Δmed={delta:+.3f} eV   |r|={abs(r_val):.3f}\n  n={n_present:,}",
            ha="center", va="top", fontsize=10,
            transform=ax_img.transAxes,
        )

    plt.savefig(fname, dpi=400, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Figure -> {fname}")


# --- Génération des figures -------------------------------------------------
for target, _ in corr.items():
    # classées par |Δmédiane|
    top_delta = delta_median[target].abs().sort_values(ascending=False).head(N_BOX).index.tolist()
    draw_boxplot_figure(
        target, top_delta,
        fname=f"ecfp4_boxplots_delta_{target}.png",
        title=f"Top {N_BOX} ECFP4 environments — {target}  (ranked by $|\\Delta$med$|$)\n(dashed line = global median)",
    )

    # classées par |r|
    top_r = corr[target].head(N_BOX).index.tolist()
    draw_boxplot_figure(
        target, top_r,
        fname=f"ecfp4_boxplots_r_{target}.png",
        title=f"Top {N_BOX} ECFP4 environments — {target}  (ranked by $|r|$)\n(dashed line = global median)",
    )

In [ ]:
# ============================================================================
#  Outliers de la régression — molécules les plus mal prédites (jeu de TEST)
#  Pour chaque cible : les N molécules avec le plus grand résidu |prédit − réel|
#  Affichage : structure 2D  +  valeur réelle / prédite / erreur
# ============================================================================
from rdkit.Chem import Draw
from IPython.display import display as ipy_display

N_OUT = 5   # molécules affichées par extrême (sur-estimées / sous-estimées)

# Reconstruire le DataFrame du jeu de test (index dans df_reg)
_, idx_test = train_test_split(np.arange(len(df_reg)), test_size=0.2, random_state=42)
df_test = df_reg.iloc[idx_test].reset_index(drop=True)

for target in TARGETS:
    r        = results[target]
    residus  = r["y_pred_te"] - r["y_te"]          # prédit − réel
    df_test  = df_test.copy()
    df_test[f"réel_{target}"]   = r["y_te"]
    df_test[f"prédit_{target}"] = r["y_pred_te"]
    df_test[f"résidu_{target}"] = residus
    df_test[f"|résidu|_{target}"] = np.abs(residus)

    rmse = r["rmse_te"]
    print(f"{'━'*62}")
    print(f"  Outliers régression — {target}  "
          f"(RMSE test = {rmse:.3f} eV, n_test = {len(df_test):,})")
    print(f"{'━'*62}")

    for label, sub in [
        (f"⬇  {N_OUT} plus SOUS-estimées  (prédit << réel)",
         df_test.nsmallest(N_OUT, f"résidu_{target}")),
        (f"⬆  {N_OUT} plus SUR-estimées   (prédit >> réel)",
         df_test.nlargest(N_OUT, f"résidu_{target}")),
    ]:
        print(f"\n{label}")
        mols    = [Chem.MolFromSmiles(s) for s in sub["SMILES"]]
        legends = [
            f"réel={row[f'réel_{target}']:.2f}\nprédit={row[f'prédit_{target}']:.2f}\nerr={row[f'résidu_{target}']:+.2f} eV"
            for _, row in sub.iterrows()
        ]
        img = Draw.MolsToGridImage(
            mols,
            molsPerRow=N_OUT,
            subImgSize=(300, 230),
            legends=legends,
        )
        ipy_display(img)

        recap = sub[["SMILES",
                     f"réel_{target}",
                     f"prédit_{target}",
                     f"résidu_{target}"]].copy()
        recap.columns = ["SMILES", f"{target} réel", f"{target} prédit", "résidu (eV)"]
        recap.index = range(1, N_OUT + 1)
        try:
            display(recap)
        except NameError:
            print(recap.to_string())
    print()